In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", data.target_names)
print("Class counts:", np.bincount(y))

Dataset shape: (569, 30)
Target shape: (569,)
Classes: ['malignant' 'benign']
Class counts: [212 357]


In [3]:
# split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # keeps the class proportions approximately consistent in both sets
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 455
Testing samples: 114


In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training mean of first feature:", X_train[:, 0].mean())
print("Training standard deviation:", X_train[:, 0].std())

Training mean of first feature: -4.337433952909183e-15
Training standard deviation: 1.0


In [ ]:

class GaussianNaiveBayes:

    def fit(self, X, y):
        self.classes = np.unique(y)

        self.means = {}
        self.variances = {}
        self.priors = {}

        for c in self.classes:
            X_c = X[y == c]

            self.means[c] = np.mean(X_c, axis=0)
            self.variances[c] = np.var(
                X_c, axis=0
            ) + 1e-9

            self.priors[c] = len(X_c) / len(X)

        return self

    def _log_likelihood(self, X, c):
        mean = self.means[c]
        var = self.variances[c]

        return -0.5 * np.sum(
            np.log(2 * np.pi * var)
            + ((X - mean) ** 2) / var,
            axis=1
        )

    def predict(self, X):
        scores = []

        for c in self.classes:
            log_prior = np.log(self.priors[c])

            log_likelihood = self._log_likelihood(
                X, c
            )

            scores.append(log_prior + log_likelihood)

        scores = np.column_stack(scores)

        return self.classes[np.argmax(scores, axis=1)]

In [ ]:
nb_model = GaussianNaiveBayes()

nb_model.fit(X_train, y_train)

nb_predictions = nb_model.predict(X_test)

print("First 20 predictions:", nb_predictions[:20])
print("First 20 actual labels:", y_test[:20])

First 20 predictions: [0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1]
First 20 actual labels: [0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 1 1 1 1]


In [8]:
accuracy = np.mean(nb_predictions == y_test)

print(f"Naive Bayes Accuracy: {accuracy:.4f}")

Naive Bayes Accuracy: 0.9298
